# 멤버 월간 소비 지표 분석

이 노트북은 월간 소비 데이터를 심층 분석하여 고정비/변동비, 반복 소비 누적, 주차별 추이, 소액 다빈도, 야간 소비, 이상 지출 누적집계, 절약 가능 금액 추정을 포함한 종합 지표를 JSON으로 출력합니다.

In [57]:
from __future__ import annotations

import json
import calendar
import pandas as pd
from pathlib import Path
from datetime import date, timedelta
from typing import Any

# ===== 분석 파라미터 =====
MEMBER_ID:        int = 1
ANALYSIS_MONTH:   str = "2024-04"  # 분석할 연-월 (YYYY-MM)

# ===== 가맹점 분류 키워드 =====
DELIVERY_KEYWORDS    = ['배달의민족', '쿠팡이츠']
CAFE_KEYWORDS        = ['스타벅스', '이디야', '투썸플레이스', '할리스', '커피빈', '폴바셋', '빽다방', '메가커피', '컴포즈']
CONVENIENCE_KEYWORDS = ['CU', 'GS25', '세븐일레븐', '미니스톱', 'emart24', '이마트24']
TAXI_KEYWORDS        = ['카카오택시', '타다', '우티']

# 고정비 판단 기준 (결제 방식)
FIXED_PAYMENT_KEYWORD = '자동이체'

# 낭비성 소비 카테고리 (변동비 중 선택적 지출)
WASTE_CATEGORIES    = ['식비', '쇼핑']
ESSENTIAL_CATEGORIES = ['교통', '의료', '생활']

MICRO_THRESHOLD: int = 10_000  # 소액 기준 (원)
LATE_NIGHT_HOUR: int = 21      # 야간 기준
SUBWAY_AVG_FARE: int = 1_400   # 택시 → 대중교통 전환 시 대당 기준 운임

print('✅ 설정 완료')

✅ 설정 완료


In [58]:
# ===== 데이터 로드 및 월 필터링 =====
df_past  = pd.read_csv(r"C:\Users\user\dev\catcher-llm\data\raw\csv\consumption_v1.csv")
df_month = pd.read_csv(r"C:\Users\user\dev\catcher-llm\notebook\team02\data_pre\data_input_month.csv")

df_past_m  = df_past[df_past['멤버 id'] == MEMBER_ID].copy()
df_month_m = df_month[df_month['멤버 id'] == MEMBER_ID].copy()

df_all = pd.concat([df_past_m, df_month_m], ignore_index=True)
df_all['사용 시간'] = pd.to_datetime(df_all['사용 시간'])
df_all['사용 금액'] = pd.to_numeric(df_all['사용 금액'])
df_all['date']    = df_all['사용 시간'].dt.date
df_all['hour']    = df_all['사용 시간'].dt.hour
df_all['weekday'] = df_all['사용 시간'].dt.dayofweek
df_all['ym']      = df_all['사용 시간'].dt.to_period('M').astype(str)

# 분석 월 / 전월 설정
year, month = int(ANALYSIS_MONTH[:4]), int(ANALYSIS_MONTH[5:])
prev_year   = year - 1 if month == 1 else year
prev_month  = 12       if month == 1 else month - 1
prev_ym     = f"{prev_year}-{prev_month:02d}"

df_this_m = df_all[df_all['ym'] == ANALYSIS_MONTH].copy()
df_prev_m = df_all[df_all['ym'] == prev_ym].copy()

# IQR 상한선 (이번 달 제외한 모든 과거 데이터 기반)
df_base = df_all[df_all['ym'] != ANALYSIS_MONTH].copy()
q1_v    = float(df_base['사용 금액'].quantile(0.25))
q3_v    = float(df_base['사용 금액'].quantile(0.75))
iqr_v   = q3_v - q1_v
ub      = q3_v + 1.5 * iqr_v

print(f"분석 월: {ANALYSIS_MONTH}  → {len(df_this_m)}건 / 총 {df_this_m['사용 금액'].sum():,.0f}원")
print(f"전   월: {prev_ym}         → {len(df_prev_m)}건 / 총 {df_prev_m['사용 금액'].sum():,.0f}원")
print(f"IQR 상한선: {ub:,.0f}원")

분석 월: 2024-04  → 165건 / 총 1,860,681원
전   월: 2024-03         → 159건 / 총 1,661,894원
IQR 상한선: 22,754원


## [1] 월간 총 지출 요약

In [59]:
this_total  = float(df_this_m['사용 금액'].sum())
prev_total  = float(df_prev_m['사용 금액'].sum())
amt_diff    = this_total - prev_total
amt_diff_r  = (amt_diff / prev_total * 100) if prev_total != 0 else 0.0

active_days = df_this_m['date'].nunique()
daily_avg   = this_total / active_days if active_days > 0 else 0.0

daily_sums  = df_this_m.groupby('date')['사용 금액'].sum()
max_day = (str(daily_sums.idxmax()), int(daily_sums.max())) if not daily_sums.empty else (None, 0)
min_day = (str(daily_sums.idxmin()), int(daily_sums.min())) if not daily_sums.empty else (None, 0)

print(f"[월간 요약]")
print(f"  총 지출  : {this_total:>12,.0f}원")
print(f"  전월 총액: {prev_total:>12,.0f}원")
print(f"  증감     : {amt_diff:>+12,.0f}원  ({amt_diff_r:+.1f}%)")
print(f"  일평균   : {daily_avg:>12,.0f}원")
print(f"  최고 지출일: {max_day[0]} ({max_day[1]:,.0f}원)")
print(f"  최저 지출일: {min_day[0]} ({min_day[1]:,.0f}원)")
print(f"  결제 건수: {len(df_this_m)}건")

[월간 요약]
  총 지출  :    1,860,681원
  전월 총액:    1,661,894원
  증감     :     +198,787원  (+12.0%)
  일평균   :       62,023원
  최고 지출일: 2024-04-01 (133,044원)
  최저 지출일: 2024-04-28 (14,566원)
  결제 건수: 165건


## [2] 고정비 vs 변동비 분석
`자동이체` 결제 방식을 고정비로 분류합니다. 구독서비스 등 정기 결제가 여기에 포함됩니다.

In [60]:
is_fixed = df_this_m['결제 방식 (온/오프라인)'].str.contains(FIXED_PAYMENT_KEYWORD, na=False)
df_fixed    = df_this_m[is_fixed].copy()
df_variable = df_this_m[~is_fixed].copy()

fixed_total    = float(df_fixed['사용 금액'].sum())
variable_total = float(df_variable['사용 금액'].sum())
fixed_ratio    = (fixed_total / this_total * 100) if this_total != 0 else 0.0

# 고정비 항목 목록
fixed_items = (
    df_fixed.groupby('결제 내역')
    .agg(count=('사용 금액', 'count'), total=('사용 금액', 'sum'))
    .reset_index()
    .sort_values('total', ascending=False)
)
fixed_items_list = [
    {'merchant': str(r['결제 내역']), 'count': int(r['count']), 'total_amount': int(r['total'])}
    for _, r in fixed_items.iterrows()
]

# 미사용 구독 후보: 고정비 항목 중 해당 기간에 관련 소비가 없는 것은 수동으로 검토 필요
print("[고정비 vs 변동비]")
print(f"  고정비: {fixed_total:>10,.0f}원  ({fixed_ratio:.1f}%)")
print(f"  변동비: {variable_total:>10,.0f}원  ({100-fixed_ratio:.1f}%)")
print(f"\n  [고정비 항목]")
for item in fixed_items_list:
    print(f"    {item['merchant']:20s}: {item['count']}회 / {item['total_amount']:,.0f}원")

[고정비 vs 변동비]
  고정비:     96,900원  (5.2%)
  변동비:  1,763,781원  (94.8%)

  [고정비 항목]
    SKT통신비              : 1회 / 65,000원
    넷플릭스                : 1회 / 17,000원
    유튜브프리미엄             : 1회 / 14,900원


## [3] 카테고리 심층 분석
낭비성 소비 / 필수 소비 구분 및 절약 가능성이 높은 카테고리를 도출합니다.

In [61]:
this_cat     = df_this_m.groupby('업종 카테고리')['사용 금액'].sum()
prev_cat     = df_prev_m.groupby('업종 카테고리')['사용 금액'].sum()
this_cat_cnt = df_this_m.groupby('업종 카테고리').size()

all_cats = sorted(set(this_cat.index) | set(prev_cat.index))
cat_deep = []
for cat in all_cats:
    ta   = float(this_cat.get(cat, 0))
    pa   = float(prev_cat.get(cat, 0))
    diff = ta - pa
    ratio = (ta / this_total * 100) if this_total != 0 else 0.0
    diff_r = (diff / pa * 100) if pa != 0 else 0.0
    cat_type = ('필수' if cat in ESSENTIAL_CATEGORIES
                else '낭비성' if cat in WASTE_CATEGORIES
                else '기타')
    cat_deep.append({
        'category':          cat,
        'type':              cat_type,
        'total_amount':      int(ta),
        'ratio_percent':     round(ratio, 4),
        'transaction_count': int(this_cat_cnt.get(cat, 0)),
        'prev_month_amount': int(pa),
        'diff_amount':       int(diff),
        'diff_rate_percent': round(diff_r, 4),
    })

cat_deep.sort(key=lambda x: x['total_amount'], reverse=True)

# 절약 가능성 TOP 3: 낭비성 카테고리 중 지출이 많은 순
top_savable = sorted(
    [r for r in cat_deep if r['type'] == '낭비성'],
    key=lambda x: x['total_amount'], reverse=True
)[:3]

print(f"{'카테고리':<8} {'구분':<6} {'이번달':>12} {'전월':>12} {'증감':>12} {'비중':>6}")
print('-' * 58)
for r in cat_deep:
    print(f"{r['category']:<8} {r['type']:<6} {r['total_amount']:>12,.0f} {r['prev_month_amount']:>12,.0f} "
          f"{r['diff_amount']:>+12,.0f} {r['ratio_percent']:>5.1f}%")

print(f"\n  [절약 가능성 TOP 3 카테고리]")
for r in top_savable:
    print(f"    {r['category']}: {r['total_amount']:,.0f}원 ({r['diff_rate_percent']:+.1f}% vs 전월)")

카테고리     구분              이번달           전월           증감     비중
----------------------------------------------------------
식비       낭비성       1,388,777    1,021,087     +367,690  74.6%
쇼핑       낭비성         175,040      365,041     -190,001   9.4%
의료       필수          107,372      116,077       -8,705   5.8%
생활       필수           96,900       65,000      +31,900   5.2%
교통       필수           92,592       94,689       -2,097   5.0%

  [절약 가능성 TOP 3 카테고리]
    식비: 1,388,777원 (+36.0% vs 전월)
    쇼핑: 175,040원 (-52.0% vs 전월)


## [4] 반복 소비 누적 분석
배달, 카페, 편의점, 택시 등 습관성 소비를 월간 누적으로 집계합니다.

In [62]:
def is_match(merchant, keywords):
    return any(kw in str(merchant) for kw in keywords)

def merch_summary_monthly(df, keywords):
    sub = df[df['결제 내역'].apply(lambda x: is_match(x, keywords))]
    return {
        'count':               int(len(sub)),
        'total_amount':        int(sub['사용 금액'].sum()),
        'avg_per_transaction': round(float(sub['사용 금액'].mean()), 2) if len(sub) > 0 else 0.0,
    }

# TOP 5 가맹점 (방문 횟수)
top5_merch = (
    df_this_m.groupby('결제 내역')
    .agg(visit_count=('사용 금액', 'count'), total_amount=('사용 금액', 'sum'))
    .reset_index()
    .sort_values('visit_count', ascending=False)
    .head(5)
)
top5_list = [
    {
        'merchant':    str(r['결제 내역']),
        'visit_count': int(r['visit_count']),
        'total_amount': int(r['total_amount']),
    }
    for _, r in top5_merch.iterrows()
]

delivery_m = merch_summary_monthly(df_this_m, DELIVERY_KEYWORDS)
cafe_m     = merch_summary_monthly(df_this_m, CAFE_KEYWORDS)
conv_m     = merch_summary_monthly(df_this_m, CONVENIENCE_KEYWORDS)
taxi_m     = merch_summary_monthly(df_this_m, TAXI_KEYWORDS)

print("[월간 반복 소비 누적]")
print(f"  배달음식: {delivery_m['count']}회 / {delivery_m['total_amount']:,.0f}원  (건당 {delivery_m['avg_per_transaction']:,.0f}원)")
print(f"  카   페: {cafe_m['count']}회 / {cafe_m['total_amount']:,.0f}원  (건당 {cafe_m['avg_per_transaction']:,.0f}원)")
print(f"  편의점 : {conv_m['count']}회 / {conv_m['total_amount']:,.0f}원")
print(f"  택   시: {taxi_m['count']}회 / {taxi_m['total_amount']:,.0f}원")
print(f"\n[TOP 5 가맹점]")
for r in top5_list:
    print(f"  {r['merchant']:15s}: {r['visit_count']}회 / {r['total_amount']:,.0f}원")

[월간 반복 소비 누적]
  배달음식: 34회 / 814,069원  (건당 23,943원)
  카   페: 51회 / 268,163원  (건당 5,258원)
  편의점 : 12회 / 88,573원
  택   시: 7회 / 70,457원

[TOP 5 가맹점]
  스타벅스           : 23회 / 121,716원
  배달의민족          : 19회 / 478,837원
  쿠팡이츠           : 15회 / 335,232원
  투썸플레이스         : 14회 / 84,818원
  이디야            : 14회 / 61,629원


## [5] 주차별 소비 추이
월을 7일 단위 주차로 나누어 소비 추이가 증가/감소 경향인지 파악합니다.

In [63]:
# 주차 계산: (일 - 1) // 7 + 1  →  1주차~5주차
df_this_m['week_num'] = df_this_m['date'].apply(lambda d: (d.day - 1) // 7 + 1)

weekly_trend_raw = (
    df_this_m.groupby('week_num')
    .agg(total_amount=('사용 금액', 'sum'), count=('사용 금액', 'count'))
    .reset_index()
)

# 주차별 시작/종료일 계산
def week_range(y, m, wn):
    start        = date(y, m, (wn - 1) * 7 + 1)
    last_day_num = calendar.monthrange(y, m)[1]
    end_day_num  = min(wn * 7, last_day_num)  # date() 생성 전에 clamping
    end          = date(y, m, end_day_num)
    return str(start), str(end)

weekly_trend_rows = []
for _, r in weekly_trend_raw.iterrows():
    wn  = int(r['week_num'])
    s, e = week_range(year, month, wn)
    weekly_trend_rows.append({
        'week_num':     wn,
        'start_date':   s,
        'end_date':     e,
        'total_amount': int(r['total_amount']),
        'count':        int(r['count']),
    })

# 추이 방향 판단 (선형)
amounts = [r['total_amount'] for r in weekly_trend_rows]
if len(amounts) >= 2:
    slope = amounts[-1] - amounts[0]
    trend = 'increasing' if slope > 0 else 'decreasing' if slope < 0 else 'stable'
else:
    trend = 'stable'

print("[주차별 소비 추이]")
max_amt = max(r['total_amount'] for r in weekly_trend_rows)
for r in weekly_trend_rows:
    bar = '█' * int(r['total_amount'] / max_amt * 25)
    print(f"  {r['week_num']}주차 ({r['start_date']} ~ {r['end_date']}): {r['total_amount']:>10,.0f}원  {bar}")
print(f"\n  소비 추이 방향: {trend}")

[주차별 소비 추이]
  1주차 (2024-04-01 ~ 2024-04-07):    561,019원  █████████████████████████
  2주차 (2024-04-08 ~ 2024-04-14):    453,982원  ████████████████████
  3주차 (2024-04-15 ~ 2024-04-21):    392,679원  █████████████████
  4주차 (2024-04-22 ~ 2024-04-28):    334,672원  ██████████████
  5주차 (2024-04-29 ~ 2024-04-30):    118,329원  █████

  소비 추이 방향: decreasing


## [6] 소액 다빈도 누적 분석
월간 소액(1만원 미만) 결제의 누적 규모를 파악합니다. 작은 금액이 쌓이면 큰 누수가 됩니다.

In [64]:
df_micro_m = df_this_m[df_this_m['사용 금액'] < MICRO_THRESHOLD].copy()

micro_total = float(df_micro_m['사용 금액'].sum())
micro_count = int(len(df_micro_m))
micro_ratio = (micro_total / this_total * 100) if this_total != 0 else 0.0

micro_cat = (
    df_micro_m.groupby('업종 카테고리')['사용 금액']
    .agg(['sum', 'count'])
    .sort_values('sum', ascending=False)
    .head(5)
)
micro_cat_list = [
    {'category': str(cat), 'total_amount': int(row['sum']), 'count': int(row['count'])}
    for cat, row in micro_cat.iterrows()
]

print(f"[소액 다빈도 누적 ({MICRO_THRESHOLD:,}원 미만)]")
print(f"  총 건수: {micro_count}건")
print(f"  총 금액: {micro_total:,.0f}원  (전체 지출 대비 {micro_ratio:.1f}%)")
print(f"  카테고리별:")
for r in micro_cat_list:
    print(f"    {r['category']}: {r['count']}건 / {r['total_amount']:,.0f}원")

[소액 다빈도 누적 (10,000원 미만)]
  총 건수: 101건
  총 금액: 538,350원  (전체 지출 대비 28.9%)
  카테고리별:
    식비: 72건 / 439,321원
    교통: 20건 / 42,077원
    의료: 5건 / 31,490원
    쇼핑: 4건 / 25,462원


## [7] 야간 소비 월간 분석
21시 이후 소비의 누적 규모와 주요 카테고리를 분석합니다.

In [65]:
df_late_m = df_this_m[df_this_m['hour'] >= LATE_NIGHT_HOUR].copy()

late_total = float(df_late_m['사용 금액'].sum())
late_count = int(len(df_late_m))
late_ratio = (late_total / this_total * 100) if this_total != 0 else 0.0

late_cat = (
    df_late_m.groupby('업종 카테고리')['사용 금액']
    .agg(['sum', 'count'])
    .sort_values('sum', ascending=False)
    .head(5)
)
late_cat_list = [
    {'category': str(cat), 'total_amount': int(row['sum']), 'count': int(row['count'])}
    for cat, row in late_cat.iterrows()
]

print(f"[야간 소비 월간 ({LATE_NIGHT_HOUR}시 이후)]")
print(f"  총 건수: {late_count}건")
print(f"  총 금액: {late_total:,.0f}원  (전체 {late_ratio:.1f}%)")
print(f"  주요 카테고리:")
for r in late_cat_list:
    print(f"    {r['category']}: {r['count']}건 / {r['total_amount']:,.0f}원")

[야간 소비 월간 (21시 이후)]
  총 건수: 33건
  총 금액: 576,886원  (전체 31.0%)
  주요 카테고리:
    식비: 25건 / 518,002원
    의료: 3건 / 32,356원
    교통: 5건 / 26,528원


## [8] 이상 지출 월간 누적 집계
일간 IQR 상한선을 기준으로 이번 달에 발생한 고액 단건 지출을 전부 누적합니다.

In [66]:
df_high_m = df_this_m[df_this_m['사용 금액'] > ub].copy()

high_total = float(df_high_m['사용 금액'].sum())
high_count = int(len(df_high_m))

high_items_m = [
    {
        'used_at':  str(r['사용 시간']),
        'merchant': str(r['결제 내역']),
        'amount':   int(r['사용 금액']),
        'category': str(r['업종 카테고리']),
    }
    for _, r in df_high_m.sort_values('사용 금액', ascending=False).iterrows()
]

print(f"[이상 지출 월간 누적 (IQR 상한 {ub:,.0f}원 초과)]")
print(f"  누적 건수: {high_count}건")
print(f"  누적 금액: {high_total:,.0f}원")
print(f"\n  내역:")
for item in high_items_m:
    print(f"    {item['used_at']} | {item['merchant']:20s} | {item['amount']:>10,.0f}원 [{item['category']}]")

[이상 지출 월간 누적 (IQR 상한 22,754원 초과)]
  누적 건수: 25건
  누적 금액: 739,516원

  내역:
    2024-04-01 10:00:00 | SKT통신비               |     65,000원 [생활]
    2024-04-07 19:27:00 | 쿠팡                   |     44,282원 [쇼핑]
    2024-04-25 09:57:00 | 쿠팡                   |     42,674원 [쇼핑]
    2024-04-12 22:58:00 | 배달의민족                |     34,439원 [식비]
    2024-04-10 21:58:00 | 배달의민족                |     34,285원 [식비]
    2024-04-30 23:18:00 | 배달의민족                |     30,666원 [식비]
    2024-04-15 18:52:00 | 배달의민족                |     29,873원 [식비]
    2024-04-12 22:45:00 | 쿠팡이츠                 |     29,761원 [식비]
    2024-04-05 18:22:00 | 배달의민족                |     28,701원 [식비]
    2024-04-26 19:58:00 | 배달의민족                |     28,540원 [식비]
    2024-04-19 13:47:00 | 쿠팡                   |     27,959원 [쇼핑]
    2024-04-19 20:44:00 | 쿠팡이츠                 |     27,272원 [식비]
    2024-04-20 19:27:00 | 배달의민족                |     27,228원 [식비]
    2024-04-20 21:58:00 | 쿠팡이츠                 |     25,860원 [식비]
    

## [9] 절약 가능 금액 추정
배달, 카페, 택시 대중교통 전환, 소액 다빈도 감축 시 예상 절약 금액을 추정합니다.

In [67]:
# 배달 30% 감축
delivery_30pct = int(delivery_m['total_amount'] * 0.30)

# 카페 격일 방문 (약 절반 감축)
cafe_half = int(cafe_m['avg_per_transaction'] * (cafe_m['count'] // 2)) if cafe_m['count'] > 1 else 0

# 택시 → 대중교통 전환 (전체 횟수의 절반을 절약 가능으로 가정)
taxi_to_transit = int((taxi_m['avg_per_transaction'] - SUBWAY_AVG_FARE) * (taxi_m['count'] // 2)) if taxi_m['count'] > 0 else 0
taxi_to_transit = max(taxi_to_transit, 0)

# 소액 소비 20% 감축
micro_20pct = int(micro_total * 0.20)

total_potential = delivery_30pct + cafe_half + taxi_to_transit + micro_20pct

# 다음 달 목표: 이번 달 대비 10% 절약
next_month_target = int(this_total * 0.90)

print("[절약 가능 금액 추정]")
print(f"  배달 30% 감축         : -{delivery_30pct:>8,.0f}원")
print(f"  카페 격일 방문        : -{cafe_half:>8,.0f}원")
print(f"  택시→대중교통 전환   : -{taxi_to_transit:>8,.0f}원")
print(f"  소액 소비 20% 감축    : -{micro_20pct:>8,.0f}원")
print(f"  ────────────────────────────")
print(f"  월간 총 절약 가능액   : -{total_potential:>8,.0f}원")
print(f"\n  다음 달 권장 목표액   :  {next_month_target:>9,.0f}원  (이번 달 대비 -10%)")

[절약 가능 금액 추정]
  배달 30% 감축         : - 244,220원
  카페 격일 방문        : - 131,452원
  택시→대중교통 전환   : -  25,995원
  소액 소비 20% 감축    : - 107,670원
  ────────────────────────────
  월간 총 절약 가능액   : - 509,337원

  다음 달 권장 목표액   :  1,674,612원  (이번 달 대비 -10%)


## [10] 현금 흐름 변동성 지수 (Cash Flow Volatility)
주차별 지출액의 변동 계수(CV)를 계산하여, 월별 페이스 조절이 잘 되고 있는지(일정한 소비) 아니면 월초/특정 주차에 과도하게 쏠려 흑자 부도 위험이 있는지 평가합니다.

In [68]:
import numpy as np

# 주차별 소비 금액 가져오기 (기존 weekly_trend_rows 활용)
weekly_amounts = [r['total_amount'] for r in weekly_trend_rows]

if len(weekly_amounts) > 0:
    mean_weekly = np.mean(weekly_amounts)
    std_weekly = np.std(weekly_amounts)
    
    # 변동 계수 (Coefficient of Variation) = 표준편차 / 평균
    # 0에 가까울수록 일정하게 소비, 높을수록 들쭉날쭉함
    cv_index = (std_weekly / mean_weekly) if mean_weekly > 0 else 0
    
    # 페이스 및 흑자 부도 위험도 평가
    if cv_index > 0.5:
        pace_status = "위험 (초반 과소비 후 후반 쪼들림 등 변동성이 매우 큼)"
    elif cv_index > 0.3:
        pace_status = "주의 (주차별 소비 편차가 꽤 있는 편)"
    else:
        pace_status = "안정 (매주 일정한 페이스로 소비 중)"

    print("[현금 흐름 변동성 지수 (Cash Flow Volatility)]")
    print(f"  주간 평균 소비액: {mean_weekly:,.0f}원")
    print(f"  주간 소비 표준편차: {std_weekly:,.0f}원")
    print(f"  변동 계수(CV): {cv_index:.2f} (0에 가까울수록 일정)")
    print(f"  페이스 진단: {pace_status}")
    print("\n  [주차별 비중]")
    for i, amt in enumerate(weekly_amounts):
        ratio = (amt / sum(weekly_amounts) * 100) if sum(weekly_amounts) > 0 else 0
        print(f"    {i+1}주차: {amt:>10,.0f}원 ({ratio:>5.1f}%)")
else:
    print("주차별 데이터가 부족합니다.")

[현금 흐름 변동성 지수 (Cash Flow Volatility)]
  주간 평균 소비액: 372,136원
  주간 소비 표준편차: 147,390원
  변동 계수(CV): 0.40 (0에 가까울수록 일정)
  페이스 진단: 주의 (주차별 소비 편차가 꽤 있는 편)

  [주차별 비중]
    1주차:    561,019원 ( 30.2%)
    2주차:    453,982원 ( 24.4%)
    3주차:    392,679원 ( 21.1%)
    4주차:    334,672원 ( 18.0%)
    5주차:    118,329원 (  6.4%)


## [11] 파레토 지출 쏠림 지수 (Spending Concentration Index)
고정비(필수 지출)를 제외한 변동비 중에서 상위 1~2개 카테고리가 차지하는 비중을 분석합니다. 특정 항목에 지출이 극단적으로 쏠려 있는지 파악하여, 절약 타겟(One-Thing)을 명확히 합니다.

In [69]:
# 변동비(필수 카테고리 제외)만 필터링
df_variable = df_this_m[~df_this_m['업종 카테고리'].isin(ESSENTIAL_CATEGORIES)]

if not df_variable.empty:
    var_total = df_variable['사용 금액'].sum()
    
    # 변동비 내 카테고리별 지출액 및 비중
    var_cat = df_variable.groupby('업종 카테고리')['사용 금액'].sum().sort_values(ascending=False)
    
    top_1_cat = var_cat.index[0]
    top_1_amt = var_cat.iloc[0]
    top_1_ratio = (top_1_amt / var_total) * 100
    
    top_2_ratio = 0
    top_2_cat = None
    if len(var_cat) > 1:
        top_2_cat = var_cat.index[1]
        top_2_amt = var_cat.iloc[1]
        top_2_ratio = (top_2_amt / var_total) * 100

    top_2_combined_ratio = top_1_ratio + top_2_ratio
    
    if top_1_ratio >= 60:
        concentration_status = "극심한 쏠림 (1위 항목 하나만 통제해도 예산 절감 효과 극대화)"
    elif top_2_combined_ratio >= 80:
        concentration_status = "파레토 쏠림 (상위 2개 항목이 전체 변동비의 80% 차지)"
    else:
        concentration_status = "분산 소비 (비교적 여러 항목에 골고루 지출 중)"

    print("[파레토 지출 쏠림 지수 (Spending Concentration Index)]")
    print(f"  총 변동비(필수제외): {var_total:,.0f}원")
    print(f"  1위 카테고리: {top_1_cat} ({top_1_amt:,.0f}원, {top_1_ratio:.1f}%)")
    if top_2_cat:
        print(f"  2위 카테고리: {top_2_cat} ({top_2_amt:,.0f}원, {top_2_ratio:.1f}%)")
    print(f"  상위 2개 합산 비중: {top_2_combined_ratio:.1f}%")
    print(f"  쏠림 진단: {concentration_status}")
else:
    print("변동비 카테고리 지출 내역이 없습니다.")

[파레토 지출 쏠림 지수 (Spending Concentration Index)]
  총 변동비(필수제외): 1,563,817원
  1위 카테고리: 식비 (1,388,777원, 88.8%)
  2위 카테고리: 쇼핑 (175,040원, 11.2%)
  상위 2개 합산 비중: 100.0%
  쏠림 진단: 극심한 쏠림 (1위 항목 하나만 통제해도 예산 절감 효과 극대화)


In [70]:
# 1. 지출 마찰력 (Frictionless Spending): 온라인/간편결제 비중
# 2. 지출 밀도 (Transaction Density): 결제 빈도(건수) vs 금액

# 온라인/간편결제 키워드
FRICTIONLESS_KEYWORDS = ['온라인', '간편결제', '앱결제', '배달']

# 마찰력 분석 (월간 데이터 df_this_m 사용)
is_frictionless = df_this_m['결제 방식 (온/오프라인)'].str.contains('|'.join(FRICTIONLESS_KEYWORDS), na=False)
df_fric = df_this_m[is_frictionless]

fric_total = float(df_fric['사용 금액'].sum())
fric_count = len(df_fric)
fric_ratio = (fric_total / this_total * 100) if this_total > 0 else 0.0

# 밀도 분석 (일평균 결제 건수 및 건당 평균 금액)
daily_counts = df_this_m.groupby('date').size()
avg_daily_count = daily_counts.mean() if not daily_counts.empty else 0
avg_per_swipe = this_total / len(df_this_m) if len(df_this_m) > 0 else 0

print("[월간 지출 마찰력 및 밀도 분석]")
print(f"  1. 지출 마찰력 (Pain of Paying)")
print(f"     - 온라인/간편결제: {fric_count}건 / {fric_total:,.0f}원")
print(f"     - 마찰력 없는 지출 비중: {fric_ratio:.1f}%")


print(f"\n  2. 지출 밀도 (Transaction Density)")
print(f"     - 하루 평균 결제 횟수: {avg_daily_count:.1f}회")
print(f"     - 1회 결제당 평균 금액: {avg_per_swipe:,.0f}원")


[월간 지출 마찰력 및 밀도 분석]
  1. 지출 마찰력 (Pain of Paying)
     - 온라인/간편결제: 81건 / 905,316원
     - 마찰력 없는 지출 비중: 48.7%

  2. 지출 밀도 (Transaction Density)
     - 하루 평균 결제 횟수: 5.5회
     - 1회 결제당 평균 금액: 11,277원


## [12] 할부 부채 압박 지수 (Installment Debt Pressure Index)
이번 달 지출 중 할부 결제가 차지하는 비중과 미래 부채 압박 정도를 분석합니다. 할부는 미래의 소득을 미리 당겨 쓰는 것이므로, 비중이 높을수록 가용 현금이 줄어드는 리스크가 있습니다.

In [73]:
# '할부 여부' 컬럼이 'Y'인 건을 할부로 판단하여 수치 지표 산출
df_install = df_this_m[df_this_m['할부 여부'] == 'Y'].copy()

install_total = df_install['사용 금액'].sum()
install_count = len(df_install)
install_ratio = (install_total / this_total * 100) if this_total > 0 else 0

print("[할부 이용 통계 지표]")
print(f"  이번 달 총 할부 결제액: {install_total:,.0f}원")
print(f"  할부 결제 건수: {install_count}건")
print(f"  전체 지출 대비 할부 비중: {install_ratio:.1f}%")

if not df_install.empty:
    # 할부 개월 수 숫자형 변환 및 평균 산출
    df_install['할부 개월'] = pd.to_numeric(df_install['할부 개월'], errors='coerce').fillna(0)
    avg_months = df_install['할부 개월'].mean()
    max_months = df_install['할부 개월'].max()
    
    print(f"  평균 할부 기간: {avg_months:.1f}개월")
    print(f"  최장 할부 기간: {int(max_months)}개월")
    
    print("\n  [주요 할부 지출 항목]")
    display(df_install[['사용 시간', '결제 내역', '사용 금액', '할부 개월']].sort_values('사용 금액', ascending=False))

[할부 이용 통계 지표]
  이번 달 총 할부 결제액: 0원
  할부 결제 건수: 0건
  전체 지출 대비 할부 비중: 0.0%
